In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/raw/StudentPerformanceFactors.csv")

display(df.head())

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [4]:
print("Kích thước dữ liệu:", df.shape)

print("\nDanh sách các cột:")
print(df.columns.tolist())

print("\nThông tin dữ liệu:")
df.info()

print("\nSố lượng giá trị thiếu mỗi cột:")
print(df.isnull().sum())

print("\nSố dòng trùng lặp:", df.duplicated().sum())

print("\nThống kê dữ liệu số:")
display(df.describe())

Kích thước dữ liệu: (6607, 20)

Danh sách các cột:
['Hours_Studied', 'Attendance', 'Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Sleep_Hours', 'Previous_Scores', 'Motivation_Level', 'Internet_Access', 'Tutoring_Sessions', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Physical_Activity', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender', 'Exam_Score']

Thông tin dữ liệu:
<class 'pandas.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Hours_Studied               6607 non-null   int64
 1   Attendance                  6607 non-null   int64
 2   Parental_Involvement        6607 non-null   str  
 3   Access_to_Resources         6607 non-null   str  
 4   Extracurricular_Activities  6607 non-null   str  
 5   Sleep_Hours                 6607 non-null   int64
 

,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Exam_Score
count,6607.000000,6607.000000,6607.00000,6607.000000,6607.000000,6607.000000,6607.000000
mean,19.975329,79.977448,7.02906,75.070531,1.493719,2.967610,67.235659
std,5.990594,11.547475,1.46812,14.399784,1.230570,1.031231,3.890456
min,1.000000,60.000000,4.00000,50.000000,0.000000,0.000000,55.000000
25%,16.000000,70.000000,6.00000,63.000000,1.000000,2.000000,65.000000
50%,20.000000,80.000000,7.00000,75.000000,1.000000,3.000000,67.000000
75%,24.000000,90.000000,8.00000,88.000000,2.000000,4.000000,69.000000
max,44.000000,100.000000,10.00000,100.000000,8.000000,6.000000,101.000000


In [5]:
missing = df.isnull().sum()

missing_summary = pd.DataFrame({
    "So_luong_thieu": missing,
    "Ty_le_thieu_%": (missing / len(df) * 100).round(2)
})

display(missing_summary[missing_summary["So_luong_thieu"] > 0])

,So_luong_thieu,Ty_le_thieu_%
Teacher_Quality,78,1.18
Parental_Education_Level,90,1.36
Distance_from_Home,67,1.01


In [6]:
X = df.drop(columns=["Exam_Score"])
y = df["Exam_Score"]

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

print("Kích thước X:", X.shape)
print("Kích thước y:", y.shape)

print("\nCác cột số:")
print(numeric_features)

print("\nCác cột phân loại:")
print(categorical_features)

print("\nKiểu dữ liệu của biến mục tiêu:")
print(y.dtype)

Kích thước X: (6607, 19)
Kích thước y: (6607,)

Các cột số:
['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Tutoring_Sessions', 'Physical_Activity']

Các cột phân loại:
['Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Motivation_Level', 'Internet_Access', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']

Kiểu dữ liệu của biến mục tiêu:
int64


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Số dòng huấn luyện:", X_train.shape[0])
print("Số dòng kiểm tra:", X_test.shape[0])
print("Đã xây dựng xong Pipeline tiền xử lý.")

Số dòng huấn luyện: 5285
Số dòng kiểm tra: 1322
Đã xây dựng xong Pipeline tiền xử lý.


In [8]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0)
}

results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Mô hình": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results)
display(results_df.round(4))

,Mô hình,MAE,RMSE,R2
0,Linear Regression,0.4524,1.8044,0.7696
1,Ridge,0.4524,1.8044,0.7697


In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge())
])

param_grid = {
    "model__alpha": [0.01, 0.1, 1, 10, 100, 1000]
}

grid_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Alpha tốt nhất:", grid_search.best_params_["model__alpha"])
print("MAE trung bình tốt nhất trên CV:", -grid_search.best_score_)

best_ridge = grid_search.best_estimator_
y_pred_ridge = best_ridge.predict(X_test)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("\nKết quả trên tập kiểm tra:")
print("MAE:", round(mean_absolute_error(y_test, y_pred_ridge), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_ridge)), 4))
print("R2:", round(r2_score(y_test, y_pred_ridge), 4))

Alpha tốt nhất: 10
MAE trung bình tốt nhất trên CV: 0.5070104864057161

Kết quả trên tập kiểm tra:
MAE: 0.4524
RMSE: 1.8041
R2: 0.7697


In [10]:
final_results = pd.DataFrame([{
    "Mô hình": "Ridge tối ưu",
    "Alpha": grid_search.best_params_["model__alpha"],
    "MAE": mean_absolute_error(y_test, y_pred_ridge),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_ridge)),
    "R2": r2_score(y_test, y_pred_ridge)
}])

display(final_results.round(4))

,Mô hình,Alpha,MAE,RMSE,R2
0,Ridge tối ưu,10,0.4524,1.8041,0.7697
